# Descrição: 
Utiliza-se de um subset contruido a partir do CMU Book Summary Dataset, um conjunto originalmente composto por 16.559 livros com resumos de enredo extraídos da Wikipedia e metadados alinhados a partir do Freebase (Bamman & Smith, 2013; Carnegie Mellon University, s.d.). Cada instância do dataset corresponde a um livro e contém sete atributos: Wikipedia ID, Freebase ID, título, autor, data de publicação, gêneros e resumo do enredo. O conjunto é disponibilizado sob a licença Creative Commons Attribution-ShareAlike (CC BY-SA) e tem sido amplamente empregado em pesquisas relacionadas à análise de texto e classificação em múltiplos gêneros literários (Risch, 2018).  
  
Do aspecto relativo a classificação de genero literário, pode-se dizer que a natureza deste dataset recai sobre um problema de classificação “multilabel”, isto é, cada livro pode pertencer a mais de uma classe literária. Existem, contudo, 2 classes gerais (que englobam algumas outras) que são antagônicas, sendo uma delas a majoritária “ficção”, e suas derivadas/relacionadas como ficção científica, por exemplo, e não ficção. Embora seja pretensioso dizer que na literatura não haja subjetividade suficiente para o contraditório, ou que em alguns contextos o que era ficção no passado já não é mais, e o que é atualmente pode não ser daqui a algum tempo, para o estudo aqui apresentado é razoável supor que um livro não é de ficção e não-ficção ao mesmo tempo. E, assim, foi criado o dataset que pode proporcionar o estudo de caso extremamente desbalanceado que se almejava. 

REF:  
- BAMMAN, D.; SMITH, N. A. New Alignment Methods for Discriminative Book Summarization. arXiv preprint arXiv:1305.1319, 2013.  
Disponível em: https://arxiv.org/abs/1305.1319. 

- CARNEGIE MELLON UNIVERSITY (CMU). CMU Book Summary Dataset. s.d.  
Disponível em: https://www.cs.cmu.edu/~dbamman/booksummaries.html. Acesso em: 30 set. 2025. 

- Link Kagle: https://www.kaggle.com/datasets/ymaricar/cmu-book-summary-dataset

# Imports & Configs

In [1]:
import zipfile
from pathlib import Path
from unidecode import unidecode
import pandas as pd
import numpy as np
import ast
import json
from sklearn.model_selection import StratifiedKFold
import os

# caminho do arquivo zip
zip_path = Path("../data/raw/archive.zip")

# pasta destino para extrair
extract_dir = Path("../data/raw")

# cria a pasta destino se não existir
extract_dir.mkdir(parents=True, exist_ok=True)

# extrai tudo
with zipfile.ZipFile(zip_path, 'r') as zip_ref:
    zip_ref.extractall(extract_dir)

print(f"Arquivos extraídos para: {extract_dir.resolve()}")

Arquivos extraídos para: /home/julio/Documentos/agents_pos/projetos/ML/ML_UFG/data/raw


In [2]:
# caminho do arquivo extraído
txt_path = Path("../data/raw/booksummaries.txt")

# ler como TSV (tab-separated)
df = pd.read_table(txt_path, header=None)
df.columns = [
    "id", 
    "freebase_id", 
    "title", 
    "author", 
    "publication_date", 
    "genres", 
    "summary"
]

# arrumar encode
# df['col_6'] = df["col_6"].str.encode("utf-8").str.decode("unicode_escape")

# Ver quantidade de unicos
for col in df.columns:
    print(f"Únicos {col}: {df[col].nunique()}")

# converter Genres de string para dict
df["genres"] = df["genres"].apply(
    lambda x: ast.literal_eval(x) if pd.notna(x) else {}
)

# criar também uma lista de gêneros
df["genres_list"] = df["genres"].apply(
    lambda g: sorted(list(g.values())) if isinstance(g, dict) else []
)

df['number_of_genres'] = df['genres_list'].apply(len)


print('shape: ', df.shape, '\n\nHEAD(2)')
display(df.head(2))

print('\n\nINFO()')
display(df.info())

Únicos id: 16559
Únicos freebase_id: 16559
Únicos title: 16277
Únicos author: 4714
Únicos publication_date: 2639
Únicos genres: 2154
Únicos summary: 16532
shape:  (16559, 9) 

HEAD(2)


,id,freebase_id,title,author,publication_date,genres,summary,genres_list,number_of_genres
0,620,/m/0hhy,Animal Farm,George Orwell,1945-08-17,"{'/m/016lj8': 'Roman à clef', '/m/06nbt': 'Sat...","Old Major, the old boar on the Manor Farm, ca...","[Children's literature, Fiction, Roman à clef,...",5
1,843,/m/0k36,A Clockwork Orange,Anthony Burgess,1962,"{'/m/06n90': 'Science Fiction', '/m/0l67h': 'N...","Alex, a teenager living in near-future Englan...","[Fiction, Novella, Satire, Science Fiction, Sp...",6




INFO()
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16559 entries, 0 to 16558
Data columns (total 9 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   id                16559 non-null  int64 
 1   freebase_id       16559 non-null  object
 2   title             16559 non-null  object
 3   author            14177 non-null  object
 4   publication_date  10949 non-null  object
 5   genres            16559 non-null  object
 6   summary           16559 non-null  object
 7   genres_list       16559 non-null  object
 8   number_of_genres  16559 non-null  int64 
dtypes: int64(2), object(7)
memory usage: 1.1+ MB


None

## Consideração
- Vimos que os IDs são unicos e Freebase_ID também. Sendo assim desconsideraremos Freebase_ID sem prejuizos.
- Genres foi passado para um formato mais propício, de lista, linpando chaves desnecessária da coluna original
    - Consideraremos apenas a lista ordenada nesse primeiro momento. Ela será passada para string para estudos sequentes e, quando necessário, retorna a lista.

- Data: Provavelmente nao será utilizada, mas em um primeiro momento, manteremos apenas o ano, para alguma analise se necessário

In [3]:
def keep_year(date):
    date_str = str(date)
    year = date_str.split('-')[0]
    return year

cond_date_notna = df['publication_date'].notna()
df.loc[cond_date_notna, 'publication_date'] = df.loc[cond_date_notna, 'publication_date'].apply(keep_year)

df['genres'] = df['genres_list'].astype(str)
df.drop(columns=['genres_list', 'freebase_id'], inplace=True)


df.head(2)

,id,title,author,publication_date,genres,summary,number_of_genres
0,620,Animal Farm,George Orwell,1945,"[""Children's literature"", 'Fiction', 'Roman à ...","Old Major, the old boar on the Manor Farm, ca...",5
1,843,A Clockwork Orange,Anthony Burgess,1962,"['Fiction', 'Novella', 'Satire', 'Science Fict...","Alex, a teenager living in near-future Englan...",6


## Tratamento para análise
- Como temos uma maneira de recuperar os dados facilmente pelos dados originais, uma vez que temos um ID único associado a cada linha, será feita um tratamento textual visando simplificar análises adiante, como identificação de duplicados
- será criada uma coluna que mede o tamanho de summary e numero de caracteres, e outra para numero de palavras
- **OBS:** Não significa que esse será o tratamento final, apenas um tratamento inicial para facilitar análises, remover ruidos, etc.

In [4]:
df.columns

Index(['id', 'title', 'author', 'publication_date', 'genres', 'summary',
       'number_of_genres'],
      dtype='object')

#

In [5]:
def clean_sr(sr_raw, use_unidecode_regex=True):
    cond_na = sr_raw.isna()
    sr_clean = sr_raw.copy().astype(str).str.lower().str.split().str.join(' ')
    if use_unidecode_regex == True:
        sr_clean = sr_clean.apply(unidecode).str.replace(r"[^\w\s]", "", regex=True)
    sr_clean = sr_clean.str.strip()
    sr_clean[cond_na] = np.nan
    return sr_clean


for col in ['title', 'author', 'summary']:
    df[col] = clean_sr(df[col])
df['genres'] = clean_sr(df['genres'], use_unidecode_regex=False)

df['len_summary_char'] = df['summary'].apply(len)
df['len_summary_words'] = df['summary'].str.split().apply(len)

df.head(2)

,id,title,author,publication_date,genres,summary,number_of_genres,len_summary_char,len_summary_words
0,620,animal farm,george orwell,1945,"[""children's literature"", 'fiction', 'roman à ...",old major the old boar on the manor farm calls...,5,5631,947
1,843,a clockwork orange,anthony burgess,1962,"['fiction', 'novella', 'satire', 'science fict...",alex a teenager living in nearfuture england l...,6,5943,998


# Análise de duplicidade
- não ha duplicados em todas colunas (obvio pois id é único)
- anomalia detectada nos envolvidos em duplicidade de "Summary"
    - nenhum resumo relevante nos 30 casos identificados (keep=False)
    - Serão desconsideradas essas amostras pois a principal informação para identificar o gênero é o summary

In [6]:
# Duplicados geral
cond_dup_geral = df.duplicated()
df = df[~cond_dup_geral].copy().reset_index(drop=True)
print('Duplicados todas colunas:', cond_dup_geral.sum())

Duplicados todas colunas: 0


In [7]:
cond_dup_summary = df.duplicated(subset=['summary'], keep=False)
print(f'Dup Summary (envolvidos): {df.duplicated(subset=["summary"]).sum()}')

display(df.loc[cond_dup_summary, 'summary'].value_counts())

# dropar duplicados de summary
df = df[~cond_dup_summary].copy().reset_index(drop=True)
print('Após remoção de duplicados de summary:', df.shape)

Dup Summary (envolvidos): 30


summary
plot outline description                                       19
62 60                                                           5
receptio                                                        3
to be added                                                     3
as described by sherryl connelly of the new york daily news     2
plot outline description62                                      2
reference                                                       2
see the articles on the separate works                          2
Name: count, dtype: int64

Após remoção de duplicados de summary: (16521, 9)


In [8]:
df.columns

Index(['id', 'title', 'author', 'publication_date', 'genres', 'summary',
       'number_of_genres', 'len_summary_char', 'len_summary_words'],
      dtype='object')

In [9]:
columns = ['title', 'author']
cond_notna = df[columns].notna().all(axis=1)
cond_dup = df[columns].duplicated(keep=False)

df[cond_notna & cond_dup].sort_values(['title', 'author', 'number_of_genres', 'len_summary_char'])

,id,title,author,publication_date,genres,summary,number_of_genres,len_summary_char,len_summary_words
5902,4849693,earthlight,arthur c clarke,1951,[],the short story details two astronomers caught...,0,407,73
1459,663048,earthlight,arthur c clarke,1955,['science fiction'],the plot describes how political tension betwe...,1,3520,563
11282,14729644,enders game,orson scott card,NaN,"['fiction', 'science fiction']",this story begins as ender is made the command...,2,1396,243
200,58901,enders game,orson scott card,1985,"['fiction', 'science fiction', 'speculative fi...",in the far future humanity has discovered inte...,3,6626,1129
13612,21624439,mr monk goes to the firehouse,lee goldberg,2006,['mystery'],a woman falls asleep while watching tv and a l...,1,231,44
4904,3698358,mr monk goes to the firehouse,lee goldberg,2006,['mystery'],adrian monk and natalie teeger stop by the uni...,1,29197,5143
9467,10312547,peter pan,j m barrie,NaN,[],the darling nursery as mr and mrs darling prep...,0,9308,1762
12706,18960109,peter pan,j m barrie,NaN,"[""children's literature"", 'fantasy', 'fiction'...",although the character appeared previously in ...,4,4575,847
7080,6035643,the keep,f paul wilson,NaN,"['fiction', 'horror', 'humour']",the keep had stood empty in the transylvanian ...,3,265,46
7076,6032387,the keep,f paul wilson,1981,"['fiction', 'horror', 'humour', 'speculative f...",german soldiers and ss einsatzkommandos alike ...,4,2679,446


#### Critério de escolha
- Nota-se que quando há duplicidade de autor e titulo e há gêneros associados, há coerência entre os generos, geralmente o maior summary traz um genero a mais que o menor. 
    - Não se considera inconsistentes esses dados.
- Quando não há genero em uma amostra, será persistida a que possui.
- Quando há generos distindos, opta-se para a que possui um maior número associado
- Quando os generos sao os mesmos, persiste a summary com mais informação

- Nesses criteirios, os ids = [4849693, 14729644, 21624439, 10312547, 6035643] serão desconsiderados

In [10]:
ids_drop = [4849693, 14729644, 21624439, 10312547, 6035643]
cond_id_drop = df['id'].isin(ids_drop)
df = df[~cond_id_drop].copy().reset_index(drop=True)
print('Após remoção de casos inconsistentes:', df.shape)

Após remoção de casos inconsistentes: (16516, 9)


## Critério mínimo de informação
- Não se sabe ao certo quantas palavras são suficientes para se definir um gênero. Porém é razoavel dizer que um número muito baixo é insuficiente.
    - Em primeira análise, vamos desconsiderar livros em que se tenha menos de 10 palavras no summary

In [11]:
n_threshold = 10
cond_len_summary = df['len_summary_words'] >= n_threshold
df = df[cond_len_summary].copy().reset_index(drop=True)
print(f'Após remoção de casos com menos de {n_threshold} palavras no summary:{df.shape}')

Após remoção de casos com menos de 10 palavras no summary:(16489, 9)


# Informações de generos

In [12]:
df_explode = df.copy()
df_explode['genre'] = df_explode['genres'].apply(eval)
df_explode = df_explode.explode('genre').reset_index(drop=True)
print('Top 20 gêneros:')
display(df_explode['genre'].value_counts().head(20))
df_explode.head()

Top 20 gêneros:


genre
fiction                   4721
speculative fiction       4286
science fiction           2853
novel                     2459
fantasy                   2400
children's literature     2117
mystery                   1379
young adult literature     823
suspense                   755
crime fiction              752
historical novel           652
thriller                   566
horror                     507
romance novel              433
historical fiction         387
detective fiction          341
adventure novel            330
non-fiction                230
alternate history          225
spy fiction                190
Name: count, dtype: int64

,id,title,author,publication_date,genres,summary,number_of_genres,len_summary_char,len_summary_words,genre
0,620,animal farm,george orwell,1945,"[""children's literature"", 'fiction', 'roman à ...",old major the old boar on the manor farm calls...,5,5631,947,children's literature
1,620,animal farm,george orwell,1945,"[""children's literature"", 'fiction', 'roman à ...",old major the old boar on the manor farm calls...,5,5631,947,fiction
2,620,animal farm,george orwell,1945,"[""children's literature"", 'fiction', 'roman à ...",old major the old boar on the manor farm calls...,5,5631,947,roman à clef
3,620,animal farm,george orwell,1945,"[""children's literature"", 'fiction', 'roman à ...",old major the old boar on the manor farm calls...,5,5631,947,satire
4,620,animal farm,george orwell,1945,"[""children's literature"", 'fiction', 'roman à ...",old major the old boar on the manor farm calls...,5,5631,947,speculative fiction


# Retirar Dados inconsistentes
- Não se pode ser fiction e não fiction ao mesmo tempo. Casos assim serão dropados

In [13]:
df_explode_fiction_non_fiction = df_explode[df_explode['genre'].astype(str).str.lower().str.contains('fiction')]
genres_fiction_non_fiction = df_explode_fiction_non_fiction['genre'].unique().tolist()
genres_fiction_non_fiction

['fiction',
 'speculative fiction',
 'science fiction',
 'utopian and dystopian fiction',
 'absurdist fiction',
 'hard science fiction',
 'gothic fiction',
 'detective fiction',
 'historical fiction',
 'non-fiction',
 'crime fiction',
 'literary fiction',
 'apocalyptic and post-apocalyptic fiction',
 'vampire fiction',
 'spy fiction',
 'comic science fiction',
 'non-fiction novel',
 'western fiction',
 'conspiracy fiction',
 'military science fiction',
 'feminist science fiction',
 'soft science fiction',
 'transgender and transsexual fiction',
 'social science fiction',
 'utopian fiction',
 'american gothic fiction',
 'urban fiction',
 'subterranean fiction',
 'superhero fiction',
 'creative nonfiction',
 'fictional crossover',
 'epic science fiction and fantasy']

In [14]:
non_fiction_genres = ['creative nonfiction', 'non-fiction novel', 'non-fiction']
fiction_genres = list(set(genres_fiction_non_fiction) - set(non_fiction_genres))

ids_fiction = df_explode[df_explode['genre'].isin(fiction_genres)]['id'].unique().tolist()
ids_non_fiction = df_explode[df_explode['genre'].isin(non_fiction_genres)]['id'].unique().tolist()
cond_fiction = df['id'].isin(ids_fiction)
cond_non_fiction = df['id'].isin(ids_non_fiction)

df_inconsistent = df[cond_fiction & cond_non_fiction].copy()
df_inconsistent

,id,title,author,publication_date,genres,summary,number_of_genres,len_summary_char,len_summary_words
57,22113,no logo,naomi klein,2000,"['anthropology', 'business', 'economics', 'fic...",the book comprises four sections no space no c...,6,5313,879
159,49759,down and out in paris and london,george orwell,1933,"['autobiographical novel', 'autobiography', 'f...",two verbless sentences introduce the scenesett...,5,6641,1184
236,61528,lost horizon,james hilton,1933,"['adventure novel', ""children's literature"", '...",the origin of the eleven numbered chapters of ...,7,3828,655
838,319373,general theory of employment interest and money,john maynard keynes,NaN,"['business', 'economics', 'fiction', 'non-fict...",the central argument of the general theory is ...,4,6869,1163
1379,620363,never cry wolf,farley mowat,1963,"[""children's literature"", 'fiction', 'nature',...",in 19481949 the canadian wildlife service assi...,4,1657,284
1488,675491,in the time of the butterflies,julia alvarez,1994,"['historical fiction', 'non-fiction novel', 'n...",this is the story of the four mirabal sisters ...,3,2140,356
1835,920695,tuesdays with morrie,mitch albom,1997,"['autobiography', 'biographical novel', 'biogr...",newspaper columnist mitch albom recounts time ...,9,1130,172
2115,1122051,into the wild,jon krakauer,1996,"['autobiography', 'biography', 'non-fiction', ...",on september 6 1992 christopher mccandlesss bo...,6,5173,856
2405,1342669,green hills of africa,ernest hemingway,1935,"['autobiography', 'biography', 'fiction', 'non...",much of narrative describes hemingways adventu...,6,1358,226
2572,1473548,the songlines,bruce chatwin,1987,"['fiction', 'non-fiction', 'travel']",in the book chatwin develops his thesis about ...,3,1547,266


In [15]:
df_inconsistent['genres'].values

array(["['anthropology', 'business', 'economics', 'fiction', 'non-fiction', 'sociology']",
       "['autobiographical novel', 'autobiography', 'fiction', 'memoir', 'non-fiction']",
       '[\'adventure novel\', "children\'s literature", \'fantasy\', \'fiction\', \'lost world\', \'non-fiction novel\', \'speculative fiction\']',
       "['business', 'economics', 'fiction', 'non-fiction']",
       '["children\'s literature", \'fiction\', \'nature\', \'non-fiction\']',
       "['historical fiction', 'non-fiction novel', 'novel']",
       "['autobiography', 'biographical novel', 'biography', 'fiction', 'inspirational', 'memoir', 'non-fiction', 'philosophy', 'sociology']",
       "['autobiography', 'biography', 'non-fiction', 'speculative fiction', 'travel', 'travel literature']",
       "['autobiography', 'biography', 'fiction', 'non-fiction', 'travel', 'travel literature']",
       "['fiction', 'non-fiction', 'travel']",
       "['fiction', 'hard science fiction', 'non-fiction']",
       "

In [16]:
inconsistent_ids = df_inconsistent['id'].unique().tolist()
df = df[~df['id'].isin(inconsistent_ids)].copy().reset_index(drop=True)

print('Após remoção de casos inconsistentes de gêneros:', df.shape)

Após remoção de casos inconsistentes de gêneros: (16473, 9)


# Separação de "sem generos rotulados"
- Existem livros sem gênero rotulado (vazio). Vamos separar esses casos dos demais, para análises futuras.
- Casos em que nao são fiction ou não fiction, mas possuem algum gênero rotulado, serão desconsiderados da análise.


In [17]:
df_not_labeled = df[df['number_of_genres'] == 0].copy().reset_index(drop=True)
ids_not_labeled = df_not_labeled['id'].unique().tolist()

# Save df_not labeled
path_blind = '../data/blind/blind_set.csv'
os.makedirs(os.path.dirname(path_blind), exist_ok=True)
df_not_labeled.to_csv(path_blind, index=False)

df_not_labeled

,id,title,author,publication_date,genres,summary,number_of_genres,len_summary_char,len_summary_words
0,1756,an enquiry concerning human understanding,david hume,NaN,[],the argument of the enquiry proceeds by a seri...,0,16591,2820
1,2950,anyone can whistle,arthur laurents,NaN,[],the story is set in an imaginary american town...,0,6990,1290
2,4331,book of joshua,NaN,NaN,[],chapter 1 is the first of three important mome...,0,3339,558
3,4332,book of ezra,NaN,NaN,[],for the bible text see bible gateway opens at ...,0,3335,588
4,4376,book of numbers,NaN,NaN,[],god orders moses in the wilderness of sinai to...,0,3983,709
...,...,...,...,...,...,...,...,...,...
3701,36531274,the birth of plenty,william j bernstein,NaN,[],the birth of plenty is an history of the world...,0,523,79
3702,36551772,telegraph avenue,michael chabon,NaN,[],set during the summer of 2004 the novel main p...,0,2818,494
3703,36665207,the simpsons a complete guide to our favorite ...,matt groening,1997,[],classwikitable seasons covered book title 18...,0,427,67
3704,36934824,under wildwood,colin meloy,2012,[],prue mckeel having rescued her brother from th...,0,940,151


In [18]:
df = df[~df['id'].isin(ids_not_labeled)].copy().reset_index(drop=True)
print('Após remoção de casos sem gênero:', df.shape)

Após remoção de casos sem gênero: (12767, 9)


# Binarização do problema extremamente desbalanceado
- Separação de casos sem gênero (removidos de treino e teste, mas separados simulando blind set)
- Remoção não classificados nem como fiction nem non-fiction (sem presunções, eles foram desconsiderados.)
- Binarização Ficção ou não Ficção


In [19]:
ids_fiction = list(set(ids_fiction) - set(inconsistent_ids))
ids_non_fiction = list(set(ids_non_fiction) - set(inconsistent_ids))

len(ids_fiction), len(ids_non_fiction)

(8685, 219)

In [20]:
# Remoção dos que não são nem fiction nem non-fiction
cond_fiction = df['id'].isin(ids_fiction)
cond_non_fiction = df['id'].isin(ids_non_fiction)
df = df[cond_fiction | cond_non_fiction].copy().reset_index(drop=True)
print('Após manter apenas fiction e non-fiction:', df.shape)
df.head(2)

Após manter apenas fiction e non-fiction: (8904, 9)


,id,title,author,publication_date,genres,summary,number_of_genres,len_summary_char,len_summary_words
0,620,animal farm,george orwell,1945,"[""children's literature"", 'fiction', 'roman à ...",old major the old boar on the manor farm calls...,5,5631,947
1,843,a clockwork orange,anthony burgess,1962,"['fiction', 'novella', 'satire', 'science fict...",alex a teenager living in nearfuture england l...,6,5943,998


In [21]:
df.drop(columns=['genres'], inplace=True)
cond_non_fiction = df['id'].isin(ids_non_fiction)

df['label'] = 0
df.loc[cond_non_fiction, 'label'] = 1
df.head()

,id,title,author,publication_date,summary,number_of_genres,len_summary_char,len_summary_words,label
0,620,animal farm,george orwell,1945,old major the old boar on the manor farm calls...,5,5631,947,0
1,843,a clockwork orange,anthony burgess,1962,alex a teenager living in nearfuture england l...,6,5943,998,0
2,986,the plague,albert camus,1947,the text of the plague is divided into five pa...,4,6473,1119,0
3,2080,a fire upon the deep,vernor vinge,NaN,the novel posits that space around the milky w...,5,4370,720,0
4,2890,a wizard of earthsea,ursula k le guin,1968,ged is a young boy on gont one of the larger i...,5,5689,1063,0


In [22]:
vc_abs = df['label'].value_counts(dropna=False)
vc_rel = df['label'].value_counts(normalize=True, dropna=False)

pd.concat([vc_abs, vc_rel], axis=1, keys=['Absolute', 'Relative'])

,Absolute,Relative
label,,
0,8685,0.975404
1,219,0.024596


In [23]:
df[cond_non_fiction].describe()

,id,number_of_genres,len_summary_char,len_summary_words,label
count,2.190000e+02,219.000000,219.000000,219.000000,219.0
mean,1.449689e+07,1.410959,2720.155251,454.771689,1.0
std,1.106409e+07,0.781124,4063.606155,697.674198,0.0
min,2.982100e+04,1.000000,91.000000,16.000000,1.0
25%,3.512235e+06,1.000000,744.500000,124.000000,1.0
50%,1.444251e+07,1.000000,1464.000000,248.000000,1.0
75%,2.335346e+07,2.000000,3376.000000,559.500000,1.0
max,3.565480e+07,5.000000,33652.000000,5716.000000,1.0


In [24]:
df[~cond_non_fiction].describe()

,id,number_of_genres,len_summary_char,len_summary_words,label
count,8.685000e+03,8685.000000,8685.000000,8685.000000,8685.0
mean,8.091843e+06,2.807484,2673.392516,468.415429,0.0
std,7.802735e+06,1.413159,2902.634321,514.050583,0.0
min,6.200000e+02,1.000000,49.000000,10.000000,0.0
25%,2.074152e+06,2.000000,751.000000,131.000000,0.0
50%,5.421043e+06,3.000000,1765.000000,307.000000,0.0
75%,1.200896e+07,4.000000,3665.000000,641.000000,0.0
max,3.715950e+07,11.000000,56451.000000,10317.000000,0.0


In [25]:
df.describe()

,id,number_of_genres,len_summary_char,len_summary_words,label
count,8.904000e+03,8904.000000,8904.000000,8904.000000,8904.000000
mean,8.249380e+06,2.773136,2674.542677,468.079852,0.024596
std,7.960328e+06,1.417614,2936.396832,519.298549,0.154898
min,6.200000e+02,1.000000,49.000000,10.000000,0.000000
25%,2.074217e+06,2.000000,751.000000,130.000000,0.000000
50%,5.487688e+06,3.000000,1756.500000,304.500000,0.000000
75%,1.227400e+07,4.000000,3659.000000,638.000000,0.000000
max,3.715950e+07,11.000000,56451.000000,10317.000000,1.000000


# Separação do conjunto de teste
- Visto que o problema a ser abordado é extemamente desbalanceado, é importante ter uma separação estratificada dos dados.
- Uma estratégia para se ter uma boa representatividade geral dos dados será definida, levando em conta não só a proporção entre as classes, mas também uma importante propriedade do dataset

## Definção da coluna de estratificação
- Deseja-se estratificar o conjunto de treino e teste levando em conta a variavel alvo (label) e uma alguma propriedade importante do dataset que permita uma boa representatividade dos dados em ambos os conjuntos.
    - O cenário ideal seria um balanceamento geral das features, porém, as features, em princípio, são não estruturadas (texto livre -> summary)
        - Extrair o vocabulário total do dataset poderia ser uma alternativa, mas as features serão tão esparsas que essa extratificação seria inviável (features por bag of words, por exemplo).
        - Considerou-se uma propriedade justa o tamanho de summary. 
            - Ele foi medido de duas formas: nro de palavras e comprimento de caracteres
                - Opta-se pelo nro de palavras, que indiretamente é uma medida do número de tokens, algo que acredita-se ser apropriado

## Definição do volume dos conjuntos
- Pensando no baixo volume da classe miniritária, não seria adequado um validação cruzada em muitos folds.
    - Optou-se pela distribuição em 6 folds, ficando o ultimo para ter o papel de conjunto de teste.

In [26]:
# Semente
seed = 2025
n_split = 6

# criar coluna de estratificação
df['strat'] = df['label'].astype(str) + '_' + pd.qcut(df['len_summary_char'], q=4, labels=False).astype(str)

# inicializar StratifiedKFold
skf = StratifiedKFold(n_splits=n_split, shuffle=True, random_state=seed)

# dividir pelos folds
for fold, (train_idx, val_idx) in enumerate(skf.split(df, df['strat'])):
    fold_name = f'fold_{fold + 1}'
    if fold == n_split - 1:
        fold_name = 'test'
    df.loc[val_idx, 'fold'] = fold_name

df.groupby(['fold', 'label'])['id'].count()

fold    label
fold_1  0        1448
        1          36
fold_2  0        1448
        1          36
fold_3  0        1448
        1          36
fold_4  0        1447
        1          37
fold_5  0        1447
        1          37
test    0        1447
        1          37
Name: id, dtype: int64

In [27]:
# Separação test set
df = df.drop(columns=['strat', 'number_of_genres', 'len_summary_char', 'len_summary_words'])
df_test = df[df['fold'] == 'test'].copy().reset_index(drop=True)
df_train_val = df[df['fold'] != 'test'].copy().reset_index(drop=True)

# Save
os.makedirs('../data/split', exist_ok=True)
df_test.to_csv('../data/split/test_set.csv', index=False)
df_train_val.to_csv('../data/split/train_val_set.csv', index=False)

print('Train/Val set:', df_train_val.shape)
print('Test set:', df_test.shape)

Train/Val set: (7420, 7)
Test set: (1484, 7)


In [28]:
ids_train_val = df_train_val['id'].unique().tolist()
ids_test = df_test['id'].unique().tolist()
print(len(set(ids_test) & set(ids_train_val)))  # deve ser vazio
print(len(set(ids_train_val) & set(ids_test))) # deve ser vazio

0
0
